
# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

In [3]:
#your code here (you can use as many cells as necessary/you prefer)

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [4]:
# mount the drive here

Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [5]:
import pandas as pd

normed_critical = pd.read_csv(r"raw-c\data\processed\normed_critical.csv")
rawc_with_dom = pd.read_csv(r"raw-c\data\processed\raw-c_with_dominance.csv")
rawc = pd.read_csv(r"raw-c\data\processed\raw-c.csv")
stims_with_nlm_dist = pd.read_csv(r"raw-c\data\processed\stims_with_nlm_distances.csv")

In [6]:
rawc.head()

,word,sentence1,sentence2,same,ambiguity_type,disambiguating_word1,disambiguating_word2,version,Class,mean_relatedness,median_relatedness,diff,count,sd_relatedness,distance_bert,distance_elmo,se_relatedness,v1,v2,string
0,act,It was a desperate act.,It was a magic act.,False,Polysemy,desperate,magic,M1_a_M2_a,N,2.181818,2.0,0.181818,11,1.328020,0.204110,0.034093,0.400413,M1_a,M2_a,act
1,act,It was a desperate act.,It was a comedic act.,False,Polysemy,desperate,comedic,M1_a_M2_b,N,2.000000,2.0,0.000000,7,1.290994,0.215616,0.045927,0.487950,M1_a,M2_b,act
2,act,It was a humane act.,It was a magic act.,False,Polysemy,humane,magic,M1_b_M2_a,N,2.818182,3.0,0.181818,11,0.981650,0.191488,0.042351,0.295979,M1_b,M2_a,act
3,act,It was a humane act.,It was a comedic act.,False,Polysemy,humane,comedic,M1_b_M2_b,N,2.809524,3.0,0.190476,21,0.928388,0.225272,0.057707,0.202591,M1_b,M2_b,act
4,act,It was a desperate act.,It was a humane act.,True,Polysemy,desperate,humane,M1_a_M1_b,N,3.900000,4.0,0.100000,10,0.316228,0.167990,0.041440,0.100000,M1_a,M1_b,act


## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

In [7]:
# install the psycho-embeddings package here

Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [8]:
import torch
from psycho_embeddings import ContextualizedEmbedder

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = ContextualizedEmbedder(
    model_name="bert-base-uncased",
    max_length=128,
    device=device,
)

print(f"Embedder ready on {device}")

c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
loading configuration file config.json from cache at C:\Users\ansar\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output

Embedder ready on cpu


Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [9]:
import numpy as np

word = "assignment"
sentence = "I am having so much fun with this assignment!"
layers = [4, 8, 12]

embeddings = embedder.embed(
    words=[word],
    target_texts=[sentence],
    layers_id=layers,
    batch_size=1,
    show_progress=False,
    averaging=True,
    return_static=True,
)

print("Returned layers:", sorted(embeddings.keys()))

for layer_id in [-1, 4, 8, 12]:
    vec = embeddings[layer_id][0]
    print(f"Layer {layer_id}: shape={vec.shape}, norm={np.linalg.norm(vec):.4f}")
    print("  first 8 values:", np.round(vec[:8], 4))

Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 229.31 examples/s]

Returned layers: [-1, 4, 8, 12]
Layer -1: shape=(768,), norm=1.2572
  first 8 values: [-0.0094 -0.0577 -0.0616 -0.0536 -0.0078 -0.112  -0.0544  0.0357]
Layer 4: shape=(768,), norm=20.9498
  first 8 values: [ 1.9968 -0.591  -0.1317 -0.5903  0.7785 -2.0171  0.6368  0.7959]
Layer 8: shape=(768,), norm=19.8974
  first 8 values: [ 1.0264 -0.6431 -0.3829 -0.276   0.2697 -1.668   0.7257  0.4577]
Layer 12: shape=(768,), norm=13.0209
  first 8 values: [ 0.3949 -0.3286 -0.1683  0.0146  0.0804 -0.8447  0.4655  0.8319]



c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [10]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np

# Rebuild final stimulus set from stimuli.csv + final raw-c.csv pairs
stimuli = pd.read_csv(r"raw-c\data\stims\stimuli.csv")

stimuli["word_norm"] = stimuli["Word"].str.lower().str.strip()
stimuli["string_norm"] = stimuli["String"].str.lower().str.strip()
rawc["word_norm"] = rawc["word"].str.lower().str.strip()
rawc["string_norm"] = rawc["string"].str.lower().str.strip()

final_pairs = rawc[["word_norm", "string_norm"]].drop_duplicates()

stimuli_final = (
    stimuli.merge(final_pairs, on=["word_norm", "string_norm"], how="inner")
    .rename(columns={"Word": "word", "String": "string"})
    [["word", "string", "M1_a", "M1_b", "M2_a", "M2_b"]]
    .sort_values(["word", "string"])
    .reset_index(drop=True)
)

stimuli_long = (
    stimuli_final
    .melt(
        id_vars=["word", "string"],
        value_vars=["M1_a", "M1_b", "M2_a", "M2_b"],
        var_name="meaning_id",
        value_name="sentence",
    )
    .sort_values(["word", "string", "meaning_id"])
    .reset_index(drop=True)
)

print(f"Final stimuli: {len(stimuli_final)}")
print(f"Sentence entries to embed (4 per stimulus): {len(stimuli_long)}")

layers = [4, 8, 12]
all_layer_ids = [-1, 4, 8, 12]


def extract_embeddings(df, batch_size=32):
    rows = df.reset_index(drop=True)
    words = rows["string"].tolist()
    texts = rows["sentence"].tolist()

    collected = {layer_id: [] for layer_id in all_layer_ids}

    for start in range(0, len(rows), batch_size):
        end = min(start + batch_size, len(rows))

        batch_out = embedder.embed(
            words=words[start:end],
            target_texts=texts[start:end],
            layers_id=layers,
            batch_size=batch_size,
            show_progress=False,
            averaging=True,
            return_static=True,
        )

        for layer_id in all_layer_ids:
            collected[layer_id].extend(batch_out[layer_id])

    records = []
    for i, row in rows.iterrows():
        records.append(
            {
                "word": row["word"],
                "string": row["string"],
                "meaning_id": row["meaning_id"],
                "sentence": row["sentence"],
                "embedding_static": collected[-1][i],
                "embedding_l4": collected[4][i],
                "embedding_l8": collected[8][i],
                "embedding_l12": collected[12][i],
            }
        )

    return records


# test
small_test = stimuli_long.head(8)
small_records = extract_embeddings(small_test, batch_size=4)
print(f"Small test completed: {len(small_records)} entries")
print("Example test entry keys:", list(small_records[0].keys()))
print("Vector shapes (static, l4, l8, l12):", small_records[0]["embedding_static"].shape, small_records[0]["embedding_l4"].shape, small_records[0]["embedding_l8"].shape, small_records[0]["embedding_l12"].shape)

Parameter 'function'=<function ContextualizedEmbedder.embed.<locals>.tokenizer_function at 0x000002454D8AA5C0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Final stimuli: 112
Sentence entries to embed (4 per stimulus): 448


Text tokenization: 100%|██████████| 4/4 [00:00<00:00, 1122.45 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 4/4 [00:00<00:00, 5087.09 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Small test completed: 8 entries
Example test entry keys: ['word', 'string', 'meaning_id', 'sentence', 'embedding_static', 'embedding_l4', 'embedding_l8', 'embedding_l12']
Vector shapes (static, l4, l8, l12): (768,) (768,) (768,) (768,)


In [11]:
# Full

embedding_records = extract_embeddings(stimuli_long, batch_size=32)
print(f"Full extraction completed: {len(embedding_records)} entries")

out_path = Path("raw-c/data/processed/rawc_contextual_embeddings.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("wb") as f:
    pickle.dump(embedding_records, f)

with out_path.open("rb") as f:
    loaded_records = pickle.load(f)

print(f"Loaded {len(loaded_records)} entries from {out_path}")

bank_records = [r for r in loaded_records if r["word"] == "bank"]
print(f"\nNumber of entries for word='bank': {len(bank_records)}")

bank_df = pd.DataFrame(bank_records)
if not bank_df.empty:
    print(
        bank_df[
            [
                "word",
                "string",
                "meaning_id",
                "sentence",
                "embedding_static",
                "embedding_l4",
                "embedding_l8",
                "embedding_l12",
            ]
        ]
    )

Text tokenization: 100%|██████████| 32/32 [00:00<00:00, 3642.07 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 32/32 [00:00<00:00, 2550.65 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 32/32 [00:00<00:00, 3844.79 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 32/32 [00:00<00:00, 3126.87 examples/

Full extraction completed: 448 entries
Loaded 448 entries from raw-c\data\processed\rawc_contextual_embeddings.pkl

Number of entries for word='bank': 4
   word  string meaning_id                   sentence  \
0  bank  banked       M1_a       He banked the plane.   
1  bank  banked       M1_b  He banked the helicopter.   
2  bank  banked       M2_a       He banked the money.   
3  bank  banked       M2_b        He banked the cash.   

                                    embedding_static  \
0  [-0.032906216, -0.023735028, -0.04939177, -0.0...   
1  [-0.032906216, -0.023735028, -0.04939177, -0.0...   
2  [-0.032906216, -0.023735028, -0.04939177, -0.0...   
3  [-0.032906216, -0.023735028, -0.04939177, -0.0...   

                                        embedding_l4  \
0  [0.26263642, -0.19250788, -0.7618483, -0.00908...   
1  [0.23791067, -0.14989159, -0.6597786, -0.06863...   
2  [0.42974025, 0.27631962, -0.3599904, 0.0248075...   
3  [0.3850463, 0.28604165, -0.25840953, 0.0243157...   


# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [ ]:
import nltk

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import wordnet as wn
from nltk.stem import PorterStemmer

ps = PorterStemmer()


def get_senses_and_glosses(word):
    """
    Return a dictionary mapping synset name -> gloss for senses whose lemma list
    contains the target word or a close derivative (via stemming).
    """
    target = word.lower().strip()
    target_stem = ps.stem(target)

    senses = {}
    for syn in wn.synsets(target):
        lemma_forms = [lemma.name().lower() for lemma in syn.lemmas()]

        keep = False
        for form in lemma_forms:
            tokens = form.replace("_", " ").split()
            if target in tokens:
                keep = True
                break
            if any(ps.stem(tok) == target_stem for tok in tokens):
                keep = True
                break

        if keep:
            senses[syn.name()] = syn.definition()

    return senses

bat_senses = get_senses_and_glosses("bat")
print(f"Number of senses kept for 'bat': {len(bat_senses)}")
for sense, gloss in bat_senses.items():
    print(f"- {sense}: {gloss}")

Number of senses kept for 'bat': 10
- bat.n.01: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
- bat.n.02: (baseball) a turn trying to get a hit
- squash_racket.n.01: a small racket with a long handle used for playing squash
- cricket_bat.n.01: the club used in playing cricket
- bat.n.05: a club used for hitting a ball in various games
- bat.v.01: strike with, or as if with a baseball bat
- bat.v.02: wink briefly
- bat.v.03: have a turn at bat
- bat.v.04: use a bat
- cream.v.02: beat thoroughly and conclusively in a competition or fight


## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [ ]:
def compute_sense_embeddings(word, batch_size=32):
    """
    Compute one embedding per WordNet sense gloss for `word`.
    Uses BERT last layer only (layer 12 for bert-base-uncased).

    Returns: list of dicts with keys: sense, gloss, embedding
    """
    senses = get_senses_and_glosses(word)
    if not senses:
        return []

    sense_ids = list(senses.keys())
    glosses = list(senses.values())

    target = word.lower().strip()
    gloss_contexts = [f"{target}: {gloss}" for gloss in glosses]
    words = [target] * len(gloss_contexts)

    layer_out = embedder.embed(
        words=words,
        target_texts=gloss_contexts,
        layers_id=[12],
        batch_size=batch_size,
        show_progress=False,
        averaging=True,
        return_static=False,
    )

    vectors = layer_out[12]

    records = []
    for i, sense_id in enumerate(sense_ids):
        records.append(
            {
                "sense": sense_id,
                "gloss": glosses[i],
                "embedding": vectors[i],
            }
        )

    return records


# test
bank_sense_embeddings = compute_sense_embeddings("bank", batch_size=32)
print(f"Number of sense embeddings for 'bank': {len(bank_sense_embeddings)}")

for rec in bank_sense_embeddings:
    print(f"\nSense: {rec['sense']}")
    print(f"Gloss: {rec['gloss']}")
    print(f"Embedding shape: {rec['embedding'].shape}")
    print(f"Embedding (first 10 values): {np.round(rec['embedding'][:10], 4)}")

print("\nFull output structure:")
print(bank_sense_embeddings)

Text tokenization: 100%|██████████| 18/18 [00:00<00:00, 2063.67 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Number of sense embeddings for 'bank': 18

Sense: bank.n.01
Gloss: sloping land (especially the slope beside a body of water)
Embedding shape: (768,)
Embedding (first 10 values): [ 0.1215 -0.1408 -0.243  -0.2461  0.4397  0.4351 -0.5008  0.5747  0.4988
 -0.3803]

Sense: depository_financial_institution.n.01
Gloss: a financial institution that accepts deposits and channels the money into lending activities
Embedding shape: (768,)
Embedding (first 10 values): [ 0.2872 -0.1348 -0.2987 -0.0554  0.5642 -0.176  -0.2764  0.4493 -0.0336
  0.0127]

Sense: bank.n.03
Gloss: a long ridge or pile
Embedding shape: (768,)
Embedding (first 10 values): [ 0.0313 -0.224  -0.5094 -0.4397  0.576   0.2878 -0.3394  0.4803  0.3952
 -0.1734]

Sense: bank.n.04
Gloss: an arrangement of similar objects in a row or in tiers
Embedding shape: (768,)
Embedding (first 10 values): [ 0.3198 -0.2358 -0.7018  0.1425  0.3217  0.2789 -0.3238  0.1181  0.2064
  0.1765]

Sense: bank.n.05
Gloss: a supply or stock held in reserve

## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

In [ ]:
from pathlib import Path
import pickle

# compute sense embeddings for each final RAW-C stimulus (word/string pair)
sense_embedding_records = []
for _, row in stimuli_final.iterrows():
    lemma = row["word"]
    lexeme = row["string"]

    senses_for_word = compute_sense_embeddings(lemma, batch_size=32)

    sense_embedding_records.append(
        {
            "word": lemma,
            "string": lexeme,
            "sense_embeddings": senses_for_word,  # each item: sense, gloss, embedding
        }
    )

print(f"Computed sense embeddings for stimuli: {len(sense_embedding_records)}")
print(f"Expected final RAW-C stimuli count: {len(stimuli_final)}")

# pickle dumpp
out_path = Path("raw-c/data/processed/rawc_sense_embeddings.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("wb") as f:
    pickle.dump(sense_embedding_records, f)

# load back to test
with out_path.open("rb") as f:
    loaded_sense_records = pickle.load(f)

print(f"Loaded {len(loaded_sense_records)} entries from {out_path}")

# test
bank_entries = [r for r in loaded_sense_records if r["word"] == "bank"]
print(f"\nNumber of entries for word='bank': {len(bank_entries)}")
if bank_entries:
    bank_entry = bank_entries[0]
    print("word:", bank_entry["word"])
    print("string:", bank_entry["string"])
    print("number of senses:", len(bank_entry["sense_embeddings"]))

    for s in bank_entry["sense_embeddings"][:3]:
        print("- sense:", s["sense"])
        print("  gloss:", s["gloss"])
        print("  embedding shape:", s["embedding"].shape)

assert len(loaded_sense_records) == len(stimuli_final), "Stimulus count mismatch."

Text tokenization: 100%|██████████| 15/15 [00:00<00:00, 2650.93 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 9/9 [00:00<00:00, 2244.14 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 6/6 [00:00<00:00, 1510.55 examples/s]
c:\Users\ansar\Desktop\CL\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 7/7 [00:00<00:00, 861.33 examples/s]
c:\U

Computed sense embeddings for stimuli: 112
Expected final RAW-C stimuli count: 112
Loaded 112 entries from raw-c\data\processed\rawc_sense_embeddings.pkl

Number of entries for word='bank': 1
word: bank
string: banked
number of senses: 18
- sense: bank.n.01
  gloss: sloping land (especially the slope beside a body of water)
  embedding shape: (768,)
- sense: depository_financial_institution.n.01
  gloss: a financial institution that accepts deposits and channels the money into lending activities
  embedding shape: (768,)
- sense: bank.n.03
  gloss: a long ridge or pile
  embedding shape: (768,)


# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [ ]:
#your code here

## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [ ]:
#your code here

Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [ ]:
#your code here

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [ ]:
#your code here

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

%your answer here

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

%your answer here

### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [ ]:
#your code here

Now, print the matches and mismatches for the static layer only.

In [ ]:
#your code here

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

%your answer here

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

%your answer here

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

%your answer here

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

%your answer here

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [ ]:
#your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [ ]:
#your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [ ]:
#your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



